
# AWR2944 DSP Interactive Workbench (Gate 12E)

> **LIMITATIONS BANNER**
> - AWR2944 real int16 ADC
> - Four physical RX channels
> - TX0 and TX1 simultaneous (Not TDM-MIMO)
> - No separable TX virtual array
> - No angle FFT or range-azimuth point cloud from this capture
> - Range and velocity axes are nominal
> - Known-distance calibration still required


In [ ]:

import warnings
warnings.filterwarnings('ignore')

import time
import json
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from pathlib import Path
from functools import lru_cache

# Production DSP package
from awr2944_dca.dsp import (
    RadarProfile, RangeProcessingConfig, DopplerProcessingConfig,
    CfarConfig, PipelineConfig, run_pipeline,
    WindowType, DCRemovalMode, ClutterRemovalMode, RxCombination
)
from awr2944_dca.dsp.pipeline import load_canonical_cube
from awr2944_dca.dsp.axes import range_axis, velocity_axis

%matplotlib inline



==================================================
1. STARTUP AND DATA VERIFICATION
==================================================


In [ ]:

# Load canonical cube
raw_path = Path("../captures/smoke_v1_headless_candidate_001/raw/smoke_v1_canonical8_Raw_0.bin")
profile = RadarProfile.from_smoke_v1()

t0 = time.perf_counter()
cube, meta = load_canonical_cube(raw_path, profile)
t_load = time.perf_counter() - t0

# Verify requirements
assert cube.shape == (8, 128, 4, 256), f"Unexpected shape: {cube.shape}"
assert cube.dtype == np.int16, f"Unexpected dtype: {cube.dtype}"

# Display metadata
print("==================================================")
print("1. STARTUP AND DATA VERIFICATION")
print("==================================================")
print(f"File Path        : {raw_path.resolve()}")
print(f"SHA256           : {meta.get('raw_sha256', 'N/A')}")
print(f"Raw Size         : {raw_path.stat().st_size if raw_path.exists() else 'N/A'} bytes")
print(f"Parser Version   : {meta.get('layout_version', 'N/A')}")
print(f"Cube Shape       : {cube.shape}")
print(f"Data Type        : {cube.dtype}")
print("-" * 50)
print("Profile Parameters:")
print(f"  Start Freq     : {profile.start_frequency_hz / 1e9:.2f} GHz")
print(f"  Slope          : {profile.slope_hz_per_s / 1e12:.3f} MHz/us")
print(f"  Sample Rate    : {profile.adc_sample_rate_hz / 1e6:.1f} Msps")
print(f"  Chirps/Frame   : {profile.chirps_per_frame}")
print(f"  Frames         : {profile.frame_count}")
print(f"  RX Count       : {profile.rx_count}")
print("-" * 50)
print(f"Range Resolution : {profile.range_resolution_m:.4f} m")
print(f"Max Range (Nom.) : {profile.max_unambiguous_range_m:.2f} m")
print(f"Vel. Resolution  : {profile.velocity_resolution_mps:.4f} m/s")
print(f"Max Vel. (Nom.)  : {profile.max_unambiguous_velocity_mps:.2f} m/s")
print("==================================================")

# Make a read-only view to prevent mutation
source_cube = cube.copy()
source_cube.flags.writeable = False



==================================================
2. INTERACTIVE CONTROLS & CACHING
==================================================


In [ ]:

# Caching decorator for performance
@lru_cache(maxsize=16)
def run_cached_pipeline(config_str):
    # Deserialize the config string (used for hashable caching)
    cfg_dict = json.loads(config_str)
    
    cfg = PipelineConfig(
        profile=profile,
        range_cfg=RangeProcessingConfig(
            window=WindowType(cfg_dict['r_win']),
            nfft=cfg_dict['r_nfft'],
            dc_removal=DCRemovalMode(cfg_dict['dc_mode'])
        ),
        doppler_cfg=DopplerProcessingConfig(
            window=WindowType(cfg_dict['d_win']),
            nfft=cfg_dict['d_nfft'],
            clutter_removal=ClutterRemovalMode(cfg_dict['clutter_mode']),
            rx_combination=RxCombination(cfg_dict['rx_comb']),
            zero_doppler_notch_bins=cfg_dict['notch_width']
        ),
        cfar_cfg=CfarConfig(
            training_cells_range=cfg_dict['cfar_tr'],
            guard_cells_range=cfg_dict['cfar_gr'],
            training_cells_doppler=cfg_dict['cfar_td'],
            guard_cells_doppler=cfg_dict['cfar_gd'],
            threshold_factor_db=cfg_dict['cfar_scale']
        )
    )
    
    t0 = time.perf_counter()
    res = run_pipeline(source_cube, cfg)
    t_process = time.perf_counter() - t0
    
    return res, t_process

# Widgets definitions
w_frame = widgets.IntSlider(value=0, min=0, max=7, description='Frame:', continuous_update=False)
w_chirp = widgets.IntSlider(value=0, min=0, max=127, description='Chirp:', continuous_update=False)
w_rx = widgets.Dropdown(options=['All', 'RX0', 'RX1', 'RX2', 'RX3'], value='RX0', description='RX View:')

w_dc = widgets.Dropdown(options=['none', 'per_chirp', 'per_rx_global'], value='per_chirp', description='DC Mode:')
w_r_win = widgets.Dropdown(options=['rectangular', 'hann', 'hamming', 'blackman', 'blackman_harris', 'kaiser'], value='hann', description='R Window:')
w_r_nfft = widgets.Dropdown(options=[256, 512, 1024], value=256, description='R NFFT:')

w_d_win = widgets.Dropdown(options=['rectangular', 'hann', 'hamming', 'blackman', 'blackman_harris', 'kaiser'], value='hann', description='D Window:')
w_d_nfft = widgets.Dropdown(options=[128, 256], value=128, description='D NFFT:')
w_clutter = widgets.Dropdown(options=['none', 'slow_time_mean', 'zero_doppler_notch'], value='slow_time_mean', description='Clutter:')
w_notch = widgets.IntSlider(value=3, min=1, max=15, step=2, description='Notch Width:', continuous_update=False)

w_rx_comb = widgets.Dropdown(options=['none', 'noncoherent_power'], value='noncoherent_power', description='RX Comb.:')
w_display = widgets.Dropdown(options=['dB', 'magnitude', 'power'], value='dB', description='Display:')
w_dyn_range = widgets.IntSlider(value=60, min=20, max=120, description='Dyn Range (dB):', continuous_update=False)
w_range_bin = widgets.IntSlider(value=10, min=0, max=128, description='Select R-Bin:', continuous_update=False)

w_cfar_tr = widgets.IntSlider(value=4, min=1, max=20, description='CFAR Train R:', continuous_update=False)
w_cfar_gr = widgets.IntSlider(value=2, min=1, max=10, description='CFAR Guard R:', continuous_update=False)
w_cfar_td = widgets.IntSlider(value=4, min=1, max=20, description='CFAR Train D:', continuous_update=False)
w_cfar_gd = widgets.IntSlider(value=2, min=1, max=10, description='CFAR Guard D:', continuous_update=False)
w_cfar_sc = widgets.FloatSlider(value=15.0, min=5.0, max=40.0, step=0.5, description='CFAR Scale:', continuous_update=False)

# Layout the controls
controls_tab = widgets.Tab()
box_layout = widgets.Layout(padding='10px', border='solid 1px #eee')

tab1 = widgets.VBox([w_frame, w_chirp, w_rx, w_display, w_dyn_range, w_range_bin], layout=box_layout)
tab2 = widgets.VBox([w_dc, w_r_win, w_r_nfft], layout=box_layout)
tab3 = widgets.VBox([w_d_win, w_d_nfft, w_clutter, w_notch, w_rx_comb], layout=box_layout)
tab4 = widgets.VBox([w_cfar_tr, w_cfar_gr, w_cfar_td, w_cfar_gd, w_cfar_sc], layout=box_layout)

controls_tab.children = [tab1, tab2, tab3, tab4]
controls_tab.set_title(0, 'Data & View')
controls_tab.set_title(1, 'Range FFT')
controls_tab.set_title(2, 'Doppler FFT')
controls_tab.set_title(3, 'CFAR')

out_log = widgets.Output(layout={'border': '1px solid black', 'height': '150px', 'overflow_y': 'auto'})



==================================================
3. WORKBENCH VIEWS (ADC, Range, Doppler, Clutter, CFAR)
==================================================


In [ ]:

out_plot = widgets.Output()
w_tabs = widgets.Tab()

def get_current_config():
    return json.dumps({
        'dc_mode': w_dc.value,
        'r_win': w_r_win.value,
        'r_nfft': w_r_nfft.value,
        'd_win': w_d_win.value,
        'd_nfft': w_d_nfft.value,
        'clutter_mode': w_clutter.value,
        'notch_width': w_notch.value,
        'rx_comb': w_rx_comb.value,
        'cfar_tr': w_cfar_tr.value,
        'cfar_gr': w_cfar_gr.value,
        'cfar_td': w_cfar_td.value,
        'cfar_gd': w_cfar_gd.value,
        'cfar_scale': w_cfar_sc.value
    }, sort_keys=True)

# Helper to format display data
def format_data(data, mode, vmin, vmax):
    if mode == 'dB':
        d = 10 * np.log10(np.maximum(np.abs(data)**2, 1e-10))
        d = np.clip(d, vmax - vmin, vmax)
        return d, f"Power (dB)"
    elif mode == 'power':
        return np.abs(data)**2, "Power (linear)"
    else:
        return np.abs(data), "Magnitude"

_updating = False

def update_dashboard(*args):
    global _updating
    if _updating:
        return
    _updating = True
    
    try:
        with out_plot:
            clear_output(wait=True)
            t_start = time.perf_counter()
        
        cfg_str = get_current_config()
        
        with out_log:
            print(f"[{time.strftime('%H:%M:%S')}] Processing pipeline...")
            
        res, t_process = run_cached_pipeline(cfg_str)
        
        f = w_frame.value
        c = w_chirp.value
        r_bin = w_range_bin.value
        rx_val = w_rx.value
        sel_rx = 0 if rx_val == 'All' else int(rx_val[-1])
        dyn_range = w_dyn_range.value
        disp = w_display.value
        active_tab = w_tabs.selected_index
        
        # Performance tracker
        t_plot_start = time.perf_counter()
        
        if active_tab == 0:
            # ----------------------------------------------------
            # RAW ADC TAB
            # ----------------------------------------------------
            fig = make_subplots(rows=3, cols=2, subplot_titles=(
                f"Raw Trace (F={f}, C={c})", "First 32 Samples",
                "Last 32 Samples", "RX Histogram",
                "Per-Chirp RMS", "Per-Frame RMS"
            ))
            
            x_samp = np.arange(256)
            for rx in range(4):
                trace = source_cube[f, c, rx, :]
                fig.add_trace(go.Scatter(x=x_samp, y=trace, name=f"RX{rx}", mode='lines'), row=1, col=1)
                fig.add_trace(go.Scatter(x=np.arange(32), y=trace[:32], name=f"RX{rx}", mode='lines', showlegend=False), row=1, col=2)
                fig.add_trace(go.Scatter(x=np.arange(224, 256), y=trace[-32:], name=f"RX{rx}", mode='lines', showlegend=False), row=2, col=1)
                fig.add_trace(go.Histogram(x=trace, name=f"RX{rx}", opacity=0.7, showlegend=False), row=2, col=2)
            
            # Per-chirp RMS
            chirp_rms = np.sqrt(np.mean(source_cube[f, :, :, :]**2, axis=-1))
            for rx in range(4):
                fig.add_trace(go.Scatter(y=chirp_rms[:, rx], name=f"RX{rx} (C-RMS)", mode='lines', showlegend=False), row=3, col=1)
                
            # Per-frame RMS
            frame_rms = np.sqrt(np.mean(source_cube[:, :, :, :]**2, axis=(1,3)))
            for rx in range(4):
                fig.add_trace(go.Scatter(y=frame_rms[:, rx], name=f"RX{rx} (F-RMS)", mode='lines', showlegend=False), row=3, col=2)
                
            fig.update_layout(height=800, title_text="Raw ADC Analysis", barmode='overlay')
            fig.show()

        elif active_tab == 1:
            # ----------------------------------------------------
            # RANGE TAB
            # ----------------------------------------------------
            fig = make_subplots(rows=2, cols=2, subplot_titles=(
                f"Range Profiles (F={f}, C={c})", "Average Range Profile (Frames)",
                "Noncoherent Sum Profile", "Range-Time Heatmap"
            ))
            
            r_axis = range_axis(profile, w_r_nfft.value)
            vmax = 10 * np.log10(np.max(np.abs(res.range_result.complex_cube)**2) + 1e-10) if disp == 'dB' else None
            
            # 1. Selected profile per RX
            for rx in range(4):
                data, lbl = format_data(res.range_result.complex_cube[f, c, rx, :], disp, vmax, dyn_range)
                fig.add_trace(go.Scatter(x=r_axis, y=data, name=f"RX{rx}",
                    hovertemplate="Range: %{x:.2f} m<br>Val: %{y:.1f}<br>RX: "+str(rx)), row=1, col=1)
                    
            # 2. Avg over frames
            avg_spec = np.mean(np.abs(res.range_result.complex_cube[:, c, :, :]), axis=0)
            for rx in range(4):
                data, _ = format_data(avg_spec[rx, :], disp, vmax, dyn_range)
                fig.add_trace(go.Scatter(x=r_axis, y=data, name=f"RX{rx} (Avg F)", showlegend=False), row=1, col=2)
                
            # 3. Noncoherent sum
            nc_sum = np.sum(np.abs(res.range_result.complex_cube[f, c, :, :])**2, axis=0)
            d_nc, _ = format_data(np.sqrt(nc_sum), disp, vmax, dyn_range)
            fig.add_trace(go.Scatter(x=r_axis, y=d_nc, name="NC-Sum", line=dict(color='black', width=2)), row=2, col=1)
            
            # 4. Range-Time Heatmap (RX0)
            rt_data = np.mean(np.abs(res.range_result.complex_cube[:, :, 0, :]), axis=1) # mean over chirps
            rt_disp, _ = format_data(rt_data, disp, vmax, dyn_range)
            time_axis = np.arange(8) * profile.frame_period_s * 1000
            fig.add_trace(go.Heatmap(z=rt_disp.T, x=time_axis, y=r_axis, colorscale='Viridis'), row=2, col=2)
            
            fig.update_layout(height=800, title_text="Range FFT Analysis")
            fig.show()

        elif active_tab == 2:
            # ----------------------------------------------------
            # RANGE-DOPPLER TAB
            # ----------------------------------------------------
            fig = make_subplots(rows=2, cols=2, specs=[[{"colspan": 2}, None], [{}, {}]],
                subplot_titles=(f"Range-Doppler Map (F={f})", f"Doppler Cut @ Range Bin {r_bin}", "Range Cut @ Doppler 0"))
            
            r_axis = range_axis(profile, w_r_nfft.value)
            v_axis = velocity_axis(profile, w_d_nfft.value)
            
            map_data = res.doppler_result.power_db[f, :, sel_rx, :]
            d_map = map_data
            if disp == 'dB':
                vmax = np.max(d_map)
                d_map = np.clip(d_map, vmax - dyn_range, vmax)
            else:
                d_map = 10**(d_map/20) # Approx back to linear for visual if requested
                
            # R-D Heatmap
            fig.add_trace(go.Heatmap(z=d_map, x=r_axis, y=v_axis, colorscale='Viridis', 
                hovertemplate="Range: %{x:.2f} m<br>Vel: %{y:.2f} m/s<br>Val: %{z:.1f}<extra></extra>"), row=1, col=1)
                
            # Doppler Cut
            fig.add_trace(go.Scatter(x=v_axis, y=d_map[:, r_bin], name="Doppler Cut"), row=2, col=1)
            
            # Range Cut
            zero_d_idx = len(v_axis) // 2
            fig.add_trace(go.Scatter(x=r_axis, y=d_map[zero_d_idx, :], name="Range Cut"), row=2, col=2)
            
            fig.update_layout(height=800, title_text="Range-Doppler Analysis")
            fig.show()

        elif active_tab == 3:
            # ----------------------------------------------------
            # CLUTTER TAB
            # ----------------------------------------------------
            # Re-run with no clutter
            cfg_no_clutter = json.loads(cfg_str)
            cfg_no_clutter['clutter_mode'] = 'none'
            res_nc, _ = run_cached_pipeline(json.dumps(cfg_no_clutter, sort_keys=True))
            
            map_nc = res_nc.doppler_result.power_db[f, :, sel_rx, :]
            map_cl = res.doppler_result.power_db[f, :, sel_rx, :]
            
            vmax = np.max(map_nc)
            map_nc = np.clip(map_nc, vmax - dyn_range, vmax)
            map_cl = np.clip(map_cl, vmax - dyn_range, vmax)
            
            fig = make_subplots(rows=2, cols=2, subplot_titles=("No Clutter Removal", "With Clutter Removal", "Range Cut (Zero Doppler)", "Doppler Cut (Max Bin)"))
            
            r_axis = range_axis(profile, w_r_nfft.value)
            v_axis = velocity_axis(profile, w_d_nfft.value)
            zero_d = len(v_axis) // 2
            
            fig.add_trace(go.Heatmap(z=map_nc, x=r_axis, y=v_axis, colorscale='Viridis'), row=1, col=1)
            fig.add_trace(go.Heatmap(z=map_cl, x=r_axis, y=v_axis, colorscale='Viridis'), row=1, col=2)
            
            fig.add_trace(go.Scatter(x=r_axis, y=map_nc[zero_d, :], name="No Clutter"), row=2, col=1)
            fig.add_trace(go.Scatter(x=r_axis, y=map_cl[zero_d, :], name="Removed"), row=2, col=1)
            
            # Find peak in removed map
            p_idx = np.unravel_index(np.argmax(map_cl), map_cl.shape)
            fig.add_trace(go.Scatter(x=v_axis, y=map_nc[:, p_idx[1]], name="No Clutter"), row=2, col=2)
            fig.add_trace(go.Scatter(x=v_axis, y=map_cl[:, p_idx[1]], name="Removed"), row=2, col=2)
            
            fig.update_layout(height=800, title_text="Clutter Removal Comparison")
            fig.show()

        elif active_tab == 4:
            # ----------------------------------------------------
            # CFAR TAB
            # ----------------------------------------------------
            fig = make_subplots(rows=2, cols=1, specs=[[{"type": "xy"}], [{"type": "table"}]], subplot_titles=(f"Range-Doppler CFAR Mask (F={f})", "Detection Table"))
            
            r_axis = range_axis(profile, w_r_nfft.value)
            v_axis = velocity_axis(profile, w_d_nfft.value)
            
            d_map = res.doppler_result.power_db[f, :, sel_rx, :]
            
            # Get detections for this frame
            if res.detections is not None:
                frame_dets = [d for d in res.detections.detections if d.frame == f]
            else:
                frame_dets = []
            
            # Plot Heatmap
            fig.add_trace(go.Heatmap(z=d_map, x=r_axis, y=v_axis, colorscale='Viridis', opacity=0.8), row=1, col=1)
            
            if frame_dets:
                rx = [d.range_m for d in frame_dets]
                ry = [d.velocity_mps for d in frame_dets]
                snr = [d.snr_db for d in frame_dets]
                text = [f"SNR: {s:.1f} dB" for s in snr]
                
                fig.add_trace(go.Scatter(x=rx, y=ry, mode='markers', 
                    marker=dict(symbol='circle-open', size=10, color='red', line_width=2),
                    text=text, name="Detections"), row=1, col=1)
                    
                # Create table
                import pandas as pd
                df = pd.DataFrame([{
                    'Range (m)': f"{d.range_m:.2f}",
                    'Velocity (m/s)': f"{d.velocity_mps:.2f}",
                    'Power (dB)': f"{d.signal_power_db:.1f}",
                    'Noise (dB)': f"{d.noise_estimate_db:.1f}",
                    'SNR (dB)': f"{d.snr_db:.1f}",
                    'R-Bin': d.range_bin,
                    'D-Bin': d.doppler_bin
                } for d in frame_dets])
                
                fig.add_trace(go.Table(
                    header=dict(values=list(df.columns), fill_color='paleturquoise', align='left'),
                    cells=dict(values=[df[k] for k in df.columns], fill_color='lavender', align='left')
                ), row=2, col=1)
            
            fig.update_layout(height=800, title_text="CFAR Processing")
            fig.show()
            
        elif active_tab == 5:
            # ----------------------------------------------------
            # PROCESSING CONFIG TAB
            # ----------------------------------------------------
            cfg_dict = json.loads(cfg_str)
            html = f"<h3>Current Processing Config</h3><pre>{json.dumps(cfg_dict, indent=2)}</pre>"
            display(HTML(html))
            
        elif active_tab == 6:
            # ----------------------------------------------------
            # EXPORT & PERFORMANCE TAB
            # ----------------------------------------------------
            t_plot = time.perf_counter() - t_plot_start
            
            html = f'''
            <h3>Performance</h3>
            <ul>
                <li>Data Load Time: <b>{t_load*1000:.1f} ms</b></li>
                <li>DSP Pipeline: <b>{t_process*1000:.1f} ms</b></li>
                <li>Plot/UI Update: <b>{t_plot*1000:.1f} ms</b></li>
            </ul>
            <h3>Export Helpers</h3>
            <p>Use the cells below to export artifacts.</p>
            '''
            display(HTML(html))
    finally:
        _updating = False

w_tabs.children = [
    widgets.VBox(), widgets.VBox(), widgets.VBox(), widgets.VBox(), 
    widgets.VBox(), widgets.VBox(), widgets.VBox()
]
w_tabs.set_title(0, '1. Raw ADC')
w_tabs.set_title(1, '2. Range')
w_tabs.set_title(2, '3. Range-Doppler')
w_tabs.set_title(3, '4. Clutter')
w_tabs.set_title(4, '5. CFAR')
w_tabs.set_title(5, '6. Config')
w_tabs.set_title(6, '7. Export/Perf')

# Observers
for w in [w_frame, w_chirp, w_rx, w_dc, w_r_win, w_r_nfft, w_d_win, w_d_nfft, 
          w_clutter, w_notch, w_rx_comb, w_display, w_dyn_range, w_range_bin,
          w_cfar_tr, w_cfar_gr, w_cfar_td, w_cfar_gd, w_cfar_sc]:
    w.observe(update_dashboard, names='value')

w_tabs.observe(update_dashboard, names='selected_index')

# Initial display
display(controls_tab)
display(w_tabs)
display(out_log)
display(out_plot)

# Trigger first render
update_dashboard()



==================================================
9. EXPORT HELPERS
==================================================


In [ ]:

import pandas as pd
from awr2944_dca.dsp.types import DetectionTable

out_dir = Path("../captures/smoke_v1_headless_candidate_001/analysis/jupyter/")
out_dir.mkdir(parents=True, exist_ok=True)

def export_current_state():
    # Fetch current state from widgets
    cfg_str = get_current_config()
    res, _ = run_cached_pipeline(cfg_str)
    
    # Export Config
    with open(out_dir / "dsp_config.json", "w") as f:
        f.write(cfg_str)
        
    # Export Detections
    det_table = DetectionTable(res.detections)
    det_table.to_csv(out_dir / "detections.csv")
    with open(out_dir / "detections.json", "w") as f:
        f.write(det_table.to_json())
        
    # Export NPZ
    np.savez_compressed(
        out_dir / "processed_arrays.npz",
        range_complex_cube=res.range_result.complex_cube,
        doppler_power_db=res.doppler_result.power_db,
        range_axis=range_axis(profile, w_r_nfft.value),
        velocity_axis=velocity_axis(profile, w_d_nfft.value)
    )
    
    print(f"Exported artifacts to: {out_dir.resolve()}")

# Uncomment to export:
# export_current_state()



==================================================
10. SMOKE TEST / VALIDATION
==================================================


In [ ]:

import traceback

def run_smoke_test():
    print("Running Smoke Test...")
    run_cached_pipeline.cache_clear()
    errors = []
    
    # Test all tabs
    for tab_idx in range(7):
        w_tabs.selected_index = tab_idx
        try:
            update_dashboard()
        except Exception as e:
            errors.append(f"Error on Tab {tab_idx}: {e}")
            traceback.print_exc()
            
    # Test clutter modes on Range-Doppler Tab
    w_tabs.selected_index = 2
    for mode in ['none', 'slow_time_mean', 'zero_doppler_notch']:
        w_clutter.value = mode
        try:
            update_dashboard()
        except Exception as e:
            errors.append(f"Error on Clutter Mode {mode}: {e}")
            traceback.print_exc()
            
    if errors:
        print("Smoke Test Failed with the following errors:")
        for err in errors:
            print(f" - {err}")
        raise RuntimeError("Notebook validation failed during widget callbacks.")
    else:
        print("Smoke Test Passed: All tabs and major configurations rendered without exception.")

run_smoke_test()
